### 1.Agent循环
循环--模型与真实世界的第一道连接。  
语言模型能推理代码，但是碰不到真实世界。不能读写文件、跑测试、看报错。  
没有循环，每次工具调用你都得手动把结果粘贴回去，你自己就是那个循环。  

In [2]:
import os
import subprocess
import json

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(base_url=os.getenv("OPENAI_BASE_URL"))


SYSTEM_PROMPT = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, do not explain."

# OpenAI 的 Tool 定义格式
TOOLS = [{
    "type": "function",
    "function": {
        "name": "bash",
        "description": "Run a shell command.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "The shell command to execute."
                }
            },
            "required": ["command"],
        }
    }
}]

def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    
    
# -- 核心代码：一个调用工具的while循环
def agent_loop(messages: list):
    while True:
        responses = client.chat.completions.create(
            model="Pro/MiniMaxAI/MiniMax-M2.5",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        message = responses.choices[0].message
        # 将助手的消息追加到历史记录中（包含工具调用信息）
        messages.append(message)
        # 检查是否需要调用工具
        if not message.tool_calls:
            # 如果没有工具调用，打印结果并结束循环
            if message.content:
                print(message.content)
            return
        
        # 处理工具调用
        tool_results = []
        for tool_call in message.tool_calls:
            if tool_call.function.name == "bash":
                # 解析参数
                args = json.loads(tool_call.function.arguments)
                command = args["command"]
                print(f"请求参数：{command}")
                output = run_bash(command)
                print(output[:200])
                # 构造工具结果信息
                tool_results.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": "bash",
                    "content": output,
                })
        
        # 将所有工具结果追加到消息历史
        messages.append(tool_results)
        

if __name__ == "__main__":
    history = []
    # OPENAI通常将System Prompt放在messages第一条
    history.append({"role": "system", "content": SYSTEM_PROMPT})
    while True:
        try:
            query = input("请输入：")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() in ("q", "exit", ""):
            break

        history.append({"role": "user", "content": query})
        agent_loop(history)
        print()




APITimeoutError: Request timed out.